# BaVarIA / BASE hierarchy demo — Location, MLP-3

This bundle is a numpy rewrite extracted from the project repository. The methods here use the same likelihood formulae and hyperparameters as the paper (the original RMIA path runs on GPU). The final cell verifies element-wise agreement against scores produced by the original code on this same bundle.

The notebook reproduces two figures at small shadow-model budgets:

1. **BaVarIA at $K = 8$** — LiRA, BASE, BaVarIA-$n$, BaVarIA-$t$, online and offline.
2. **BASE hierarchy at $K = 4$ and $K = 8$** — BASE1–BASE4 + RMIA, online.

Each replicate holds out one antithetic pair of shadow models as the audit target, samples $K/2$ pairs from the remaining 4 as shadows, and scores all 5,010 population samples. ROC curves are interpolated onto a common log-spaced FPR grid and averaged across replicates.

This is a reduced version of the bundle: only **5 antithetic pairs** of shadow models are shipped (down from 33), so the maximum reachable $K$ is 8. The full pair pool is held in the project repository.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import matplotlib.pyplot as plt

import methods as m
from utils import load_bundle, sample_pairs, metrics, interp_roc

os.makedirs('figures', exist_ok=True)
bundle = load_bundle()
for k, v in bundle['meta'].items():
    print(f'{k}: {v}')
print(f"shadow_logits.shape = {bundle['shadow_logits'].shape}")

In [ ]:
FPR_GRID = np.logspace(-4, 0, 500)
N_PAIRS = bundle['meta']['n_pairs']


def replicates_for_K(K, n_reps, seed=0):
    """Yield (target_pair, shadow_idx) for each replicate."""
    rng = np.random.default_rng(seed)
    pairs_to_use = K // 2
    for rep in range(n_reps):
        target_pair = rep % N_PAIRS
        shadow_idx = sample_pairs(N_PAIRS, pairs_to_use, rng, exclude_pair=target_pair)
        yield target_pair, shadow_idx


def gather_signals(target_pair, shadow_idx):
    """Pull target+shadow tensors and pre-compute confidence/loss."""
    target_idx = 2 * target_pair  # use first model of the held-out pair as target
    t_logits = bundle['shadow_logits'][:, target_idx]
    t_mask = bundle['shadow_mask'][:, target_idx]
    s_logits = bundle['shadow_logits'][:, shadow_idx]
    s_mask = bundle['shadow_mask'][:, shadow_idx]
    t_conf = m.confidence_from_logits(t_logits)
    t_loss = m.loss_from_confidence(t_conf)
    s_conf = m.confidence_from_logits(s_logits)
    s_loss = m.loss_from_confidence(s_conf)
    return dict(t_logits=t_logits, t_mask=t_mask, t_conf=t_conf, t_loss=t_loss,
                s_logits=s_logits, s_mask=s_mask, s_conf=s_conf, s_loss=s_loss)

## Figure 1 — BaVarIA vs LiRA / BASE at $K = 8$

In [ ]:
K_BAVARIA = 8
N_REPS_BAVARIA = 5  # paper uses 32; halved here to keep the notebook fast

BAVARIA_METHODS = ['LiRA', 'BASE', 'BaVarIA-n', 'BaVarIA-t']
BAVARIA_COLORS = {'LiRA': '#1f77b4', 'BASE': '#2ca02c',
                  'BaVarIA-n': '#e377c2', 'BaVarIA-t': '#d62728'}


def score_bavaria_panel(d, online):
    return {
        'LiRA':      m.lira(d['t_logits'], d['s_logits'], d['s_mask'], online=online),
        'BASE':      m.base1(d['t_loss'], d['s_loss'], d['s_mask'], online=online),
        'BaVarIA-n': m.bavaria_n(d['t_logits'], d['s_logits'], d['s_mask'], online=online),
        'BaVarIA-t': m.bavaria_t(d['t_logits'], d['s_logits'], d['s_mask'], online=online),
    }


rocs = {setting: {meth: [] for meth in BAVARIA_METHODS} for setting in ('online', 'offline')}
aucs = {setting: {meth: [] for meth in BAVARIA_METHODS} for setting in ('online', 'offline')}

for tp, sidx in replicates_for_K(K_BAVARIA, N_REPS_BAVARIA, seed=0):
    d = gather_signals(tp, sidx)
    for setting in ('online', 'offline'):
        scores = score_bavaria_panel(d, online=(setting == 'online'))
        for meth, sc in scores.items():
            rocs[setting][meth].append(interp_roc(d['t_mask'], sc, FPR_GRID))
            aucs[setting][meth].append(metrics(d['t_mask'], sc)['AUC'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, setting in zip(axes, ('online', 'offline')):
    for meth in BAVARIA_METHODS:
        mean_tpr = np.mean(rocs[setting][meth], axis=0)
        mean_auc = float(np.mean(aucs[setting][meth]))
        ax.plot(FPR_GRID, mean_tpr, color=BAVARIA_COLORS[meth], linewidth=1.5,
                label=f'{meth} ({mean_auc:.3f})')
    ax.plot([1e-4, 1], [1e-4, 1], 'k--', linewidth=0.5, alpha=0.4)
    ax.set(xscale='log', yscale='log', xlim=(1e-4, 1), ylim=(1e-4, 1),
           xlabel='FPR', ylabel='TPR' if setting == 'online' else None,
           title=f'Location / MLP3 — {setting.capitalize()}, $K = {K_BAVARIA}$')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.25, which='major')
fig.tight_layout()
fig.savefig('figures/bavaria_roc_K8.png', dpi=140, bbox_inches='tight')
plt.show()

## Figure 2 — BASE hierarchy at $K = 4$ and $K = 8$

BASE1 (pooled exponential), BASE2 (pooled Gaussian), BASE3 (separate means / pooled variance), BASE4 (class-conditional Gaussian; equivalent to LiRA online), and RMIA. The paper also shows $K = 64$ and $K = 254$; those need a larger pair pool than the 5 shipped here.

In [ ]:
BASE_METHODS = ['BASE1', 'BASE2', 'BASE3', 'BASE4', 'RMIA']
BASE_COLORS = {
    'BASE1': '#2ca02c', 'BASE2': '#ff7f0e', 'BASE3': '#9467bd',
    'BASE4': '#1f77b4', 'RMIA':  '#d62728',
}
K_BASE = [4, 8]
N_REPS_BASE = 5


def score_base_panel(d):
    return {
        'BASE1': m.base1(d['t_loss'], d['s_loss'], d['s_mask'], online=True),
        'BASE2': m.base2(d['t_logits'], d['s_logits'], d['s_mask'], online=True),
        'BASE3': m.base3(d['t_logits'], d['s_logits'], d['s_mask']),
        'BASE4': m.base4(d['t_logits'], d['s_logits'], d['s_mask']),
        'RMIA':  m.rmia(d['t_conf'], d['s_conf'], d['s_mask'], online=True),
    }


panels = {K: {meth: {'roc': [], 'auc': []} for meth in BASE_METHODS} for K in K_BASE}
for K in K_BASE:
    for tp, sidx in replicates_for_K(K, N_REPS_BASE, seed=K):
        d = gather_signals(tp, sidx)
        scores = score_base_panel(d)
        for meth, sc in scores.items():
            panels[K][meth]['roc'].append(interp_roc(d['t_mask'], sc, FPR_GRID))
            panels[K][meth]['auc'].append(metrics(d['t_mask'], sc)['AUC'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, K in zip(axes, K_BASE):
    for meth in BASE_METHODS:
        mean_tpr = np.mean(panels[K][meth]['roc'], axis=0)
        mean_auc = float(np.mean(panels[K][meth]['auc']))
        ax.plot(FPR_GRID, mean_tpr, color=BASE_COLORS[meth], linewidth=1.5,
                label=f'{meth} ({mean_auc:.3f})')
    ax.plot([1e-4, 1], [1e-4, 1], 'k--', linewidth=0.5, alpha=0.4)
    ax.set(xscale='log', yscale='log', xlim=(1e-4, 1), ylim=(1e-4, 1),
           xlabel='FPR', ylabel='TPR' if K == K_BASE[0] else None,
           title=f'Location / MLP3 — BASE hierarchy, $K = {K}$ (online)')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.25, which='major')
fig.tight_layout()
fig.savefig('figures/base_hierarchy_K4_K8.png', dpi=140, bbox_inches='tight')
plt.show()

## Summary table

In [ ]:
rows = []
for setting in ('online', 'offline'):
    for meth in BAVARIA_METHODS:
        rows.append((f'BaVarIA panel — {setting}', meth, K_BAVARIA,
                     float(np.mean(aucs[setting][meth]))))
for K in K_BASE:
    for meth in BASE_METHODS:
        rows.append(('BASE hierarchy — online', meth, K,
                     float(np.mean(panels[K][meth]['auc']))))

print(f"{'panel':<28} {'method':<12} {'K':>4}   AUC")
for panel, meth, K, auc in rows:
    print(f'{panel:<28} {meth:<12} {K:>4}   {auc:.4f}')

## Verification — numpy demo vs the original pipeline

Sanity check that the cleaned-up methods in `methods.py` produce the same scores as the implementations we ran for the paper. `data/reference_scores_K8.npz` was generated by the original code on this same 5-pair bundle, for a fixed `(target_pair, shadow_idx)` configuration. The cell below re-scores that configuration with the demo methods and compares element-wise.

`OK` on every line means the demo and the pipeline agree to float32 round-off.

In [ ]:
ref = np.load('data/reference_scores_K8.npz')
target_pair = int(ref['target_pair'])
shadow_idx = ref['shadow_idx']
K = int(ref['K'])

target_idx = 2 * target_pair
t_logits = bundle['shadow_logits'][:, target_idx]
t_conf = m.confidence_from_logits(t_logits)
t_loss = m.loss_from_confidence(t_conf)
s_logits = bundle['shadow_logits'][:, shadow_idx]
s_mask = bundle['shadow_mask'][:, shadow_idx]
s_conf = m.confidence_from_logits(s_logits)
s_loss = m.loss_from_confidence(s_conf)

demo_scores = {
    'LiRA_online':       m.lira(t_logits, s_logits, s_mask, online=True),
    'LiRA_offline':      m.lira(t_logits, s_logits, s_mask, online=False),
    'BASE1_online':      m.base1(t_loss, s_loss, s_mask, online=True),
    'BASE1_offline':     m.base1(t_loss, s_loss, s_mask, online=False),
    'BaVarIA-n_online':  m.bavaria_n(t_logits, s_logits, s_mask, online=True),
    'BaVarIA-n_offline': m.bavaria_n(t_logits, s_logits, s_mask, online=False),
    'BaVarIA-t_online':  m.bavaria_t(t_logits, s_logits, s_mask, online=True),
    'BaVarIA-t_offline': m.bavaria_t(t_logits, s_logits, s_mask, online=False),
}

print(f'Verifying against pipeline scores (target_pair={target_pair}, K={K})')
print(f'{"method":<22} {"max |demo - ref|":>20}    status')
all_ok = True
for key, demo in demo_scores.items():
    ref_arr = ref[f'scores_{key}']
    diff = np.max(np.abs(demo.astype(np.float32) - ref_arr))
    ok = np.allclose(demo.astype(np.float32), ref_arr, atol=1e-4)
    status = 'OK' if ok else 'MISMATCH'
    all_ok &= ok
    print(f'{key:<22} {diff:>20.3e}    {status}')
assert all_ok, 'numpy demo diverges from the GPU pipeline'